# Running on real hardware

This notebook executes the measurement protocol on a superconducting
processor. The claim under test is operational: the number of distinct
circuits a device must run is **three**, whatever the problem size, because
the Pauli strings partition into three mutually commuting families.

Stages 1 and 2 are free and offline. Stage 3 onwards submits jobs and costs
credits, so each is a separate cell you have to run deliberately.

In [ ]:
%pip install -q -e ..[hardware]
from qgridx.experiments import hardware
import qgridx
print("qGridX", qgridx.__version__)

## 1. Prepare the circuits (offline, free)

Trains the circuits and freezes their tokens, string assignment and exact
correlators to a manifest. Nothing touches a device.

In [ ]:
hardware.prepare()

## 2. Verify the emission (offline, free)

**Do not skip this.** A device disagreement only tells you something if the
program you sent is provably the circuit you meant. This check executes the
generated tokens two ways, through the package executor and by interpreting
the emitted QASM, and requires them to agree to numerical precision.

It also checks each basis rotation reads the family it claims to, which is
what settled the sign question on the single-gate Y rotation.

In [ ]:
hardware.validate()

## 3. Probe the cost (submits ONE job)

Submits a single job and reports what it actually cost, so the campaign size
is chosen from a measurement rather than an estimate.

In [ ]:
# hardware.probe()          # uncomment to submit

## 4. Run the campaign (submits jobs)

Three measurement settings per circuit plus read-out calibration. The call is
resumable and skips anything already recorded, so an interrupted run is
continued rather than restarted.

If a poll is interrupted after submission, use `hardware.collect(label, axis,
job_id)` instead of rerunning. The result is already on the server and
resubmitting it costs credits for nothing.

In [ ]:
# hardware.run()            # uncomment to submit the campaign

## 5. Analyze

Reconstructs every correlator from the three measurement records and scores
sign agreement and magnitude retention against exact simulation. Both bit
orderings are reported, so the convention is visible rather than assumed.

In [ ]:
hardware.analyze()

## What the reported campaign measured

| | m = 45, n = 6 | m = 105, n = 7 |
|---|---|---|
| Device jobs | **3** | **3** |
| Shots | **3,072** | **3,072** |
| Correlators recovered | **45 of 45** | **105 of 105** |
| Magnitude retained | 5.1% | 2.5% |

More than doubling the decision count changed neither the job count nor the
shot budget. That is the encoding's central operational property, measured
rather than argued.

What the device returns at these circuit depths is reported plainly: 59 and 68
generated gates expand to 90 and 118 controlled-NOTs, against a measured
read-out error of about 6% per qubit, and the correlator magnitudes do not
survive. A two-qubit Bell control in the same session got 24 of 24 signs
correct, which places the shortfall in accumulated circuit error rather than
in the encoding or the read-out pipeline.